In [ ]:
from arize import ArizeClient
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")
DATASET_ID = "RGF0YXNldDozNDE5OTU6dGZ4YQ==" # taken from 02_dataset.ipynb

import os, certifi
CA = certifi.where()

os.environ["SSL_CERT_FILE"] = CA
os.environ["REQUESTS_CA_BUNDLE"] = CA
os.environ["CURL_CA_BUNDLE"] = CA

print("Using CA:", CA)


In [ ]:
client = ArizeClient(api_key=API_KEY, request_verify=False)

In [ ]:
ds = client.datasets.list_examples(dataset=DATASET_ID, space=SPACE_ID)
examples = ds.to_dict()["examples"]
examples

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-2025-04-14", temperature=0.0)
chain = llm | StrOutputParser()

# t
def sample_task(dataset_row) -> str:
  return chain.invoke(dataset_row.get("question"))

In [ ]:
sample_task(examples[0])

In [ ]:
from phoenix.evals import llm_classify, OpenAIModel
from arize.experiments import EvaluationResult
import pandas as pd

eval_prompt_template = """
In this task, you will be presented with a query, some context and a response. The response
is generated to the question based on the context. The response may contain false
information. You must use the context to determine if the response to the question
contains false information, if the response is a hallucination of facts. Your objective is
to determine whether the response text contains factual information and is not a
hallucination. A 'hallucination' refers to a response that is not based on the context or
assumes information that is not available in the context. Your response should be a single
word: either 'factual' or 'hallucinated', and it should not include any other text or
characters. 'hallucinated' indicates that the response provides factually inaccurate
information to the query based on the context. 'factual' indicates that the response to
the question is correct relative to the context, and does not contain made up
information. Please read the query and context carefully before determining your
response.

[BEGIN DATA]
************
[Query]: {input}
************
[Response]: {output}
************
[END DATA]

Is the response above factual or hallucinated based on the query and context?
"""

def correctness_eval(output, dataset_row):
    eval_df = llm_classify(
        dataframe=pd.DataFrame([{"input": dataset_row.get("question"), "output": output}]),
        template=eval_prompt_template,
        model=OpenAIModel(model="gpt-4o-mini"),
        rails=["correct", "incorrect"],
        provide_explanation=True,
    )

    # Map the eval df to EvaluationResult
    label = eval_df["label"][0]
    score = 1 if label == "correct" else 0
    explanation = eval_df["explanation"][0]

    return EvaluationResult(label=label, score=score, explanation=explanation)

In [ ]:
experiment, experiment_df = client.experiments.run(
    name="llm-eval-experiment",
    dataset=DATASET_ID,
    task=sample_task,
    evaluators=[correctness_eval],
)